## Setup
Generated by `scripts/build_notebooks.py`; do not edit. Installs `whopays` at `WHOPAYS_REF`,
reads secrets, and prints the environment. Needs a GPU runtime (Colab: Runtime → Change
runtime type → T4 GPU. Kaggle: Settings → Accelerator → GPU T4 x2, and Internet on).

In [ ]:
WHOPAYS_REF = "main"  # @param {type:"string"}
MOUNT_DRIVE = False  # @param {type:"boolean"}
# Offline only: exact path of a wheel from `build_notebooks.py --bundle`, e.g.
# /content/whopays-0.1.0-py3-none-any.whl (no quotes in the form).
# Empty = install from GitHub at WHOPAYS_REF.
WHOPAYS_WHEEL = ""  # @param {type:"string"}
REPO = "SVamseekar/whopays"

import os, subprocess, sys  # noqa: E401

# Colab string fields can preserve quotation marks pasted with a Python path literal.
WHOPAYS_WHEEL = WHOPAYS_WHEEL.strip()
if (len(WHOPAYS_WHEEL) >= 2 and WHOPAYS_WHEEL[0] in "\"'"
        and WHOPAYS_WHEEL[-1] == WHOPAYS_WHEEL[0]):
    WHOPAYS_WHEEL = WHOPAYS_WHEEL[1:-1]

PLATFORM = ("colab" if "COLAB_RELEASE_TAG" in os.environ
            else "kaggle" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else "local")
_gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True)
if _gpu.returncode != 0:
    raise SystemExit("No CUDA GPU attached. Colab: Runtime -> Change runtime type -> T4 GPU. "
                     "Kaggle: Settings -> Accelerator -> GPU T4 x2.")
print("platform:", PLATFORM, "| GPU:", _gpu.stdout.strip().replace("\n", ", "))


def _secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        if PLATFORM == "colab":
            from google.colab import userdata
            return userdata.get(name)
        if PLATFORM == "kaggle":
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
    except Exception as exc:
        print(f"secret {name} not available ({type(exc).__name__})")
    return None


def _sh(*cmd, **kw):
    subprocess.run(cmd, check=True, **kw)


for _name in ("HF_TOKEN", "GH_TOKEN"):
    if (_value := _secret(_name)):
        os.environ[_name] = _value

if WHOPAYS_WHEEL:  # opt-in offline fallback: exact path to a wheel you built and uploaded
    if not (os.path.isfile(WHOPAYS_WHEEL) and os.path.basename(WHOPAYS_WHEEL).startswith("whopays-")
            and WHOPAYS_WHEEL.endswith(".whl")):
        raise SystemExit(f"WHOPAYS_WHEEL={WHOPAYS_WHEEL!r} is not a whopays-*.whl file")
    import hashlib, zipfile  # noqa: E401
    with zipfile.ZipFile(WHOPAYS_WHEEL) as _archive:
        _locks = [n for n in _archive.namelist()
                  if n.endswith(".data/data/share/whopays/requirements.lock")]
        if len(_locks) != 1:
            raise SystemExit("Wheel has no unique embedded requirements.lock; rebuild it")
        _lock = _archive.read(_locks[0]).decode("utf-8")
    with open(WHOPAYS_WHEEL, "rb") as _wheel:
        _wheel_code = "wheel:sha256:" + hashlib.file_digest(_wheel, "sha256").hexdigest()
    if "whopays" in sys.modules and os.environ.get("WHOPAYS_CODE_VERSION") != _wheel_code:
        raise SystemExit("Restart the runtime before loading a different WhoPays build")
    _keep = ("numpy==", "pandas==", "pyarrow==", "scipy==", "matplotlib==")
    with open("/tmp/whopays-constraints.txt", "w") as _constraints:
        _constraints.writelines(line for line in _lock.splitlines(keepends=True)
                                if not line.startswith(_keep))
    _sh(sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/whopays-constraints.txt",
        WHOPAYS_WHEEL + "[gpu]")
    # pip can skip a rebuilt wheel with the same package version. Reinstall only our code.
    _sh(sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
        WHOPAYS_WHEEL)
    os.environ["WHOPAYS_CODE_VERSION"] = _wheel_code
else:
    _src = "/content/whopays" if PLATFORM == "colab" else "/kaggle/working/whopays-src"
    if os.environ.get("GH_TOKEN"):  # private repo; keeps the token out of URLs and logs
        os.environ.update(GIT_CONFIG_COUNT="1",
                          GIT_CONFIG_KEY_0=f"url.https://x-access-token:{os.environ['GH_TOKEN']}"
                                           "@github.com/.insteadOf",
                          GIT_CONFIG_VALUE_0="https://github.com/")
    if not os.path.isdir(_src):
        _sh("git", "init", "-q", _src)
        _sh("git", "-C", _src, "remote", "add", "origin", f"https://github.com/{REPO}.git")
    _sh("git", "-C", _src, "fetch", "-q", "--depth", "1", "origin", WHOPAYS_REF)
    _sh("git", "-C", _src, "checkout", "-q", "--force", "FETCH_HEAD")
    _commit = subprocess.run(["git", "-C", _src, "rev-parse", "HEAD"], capture_output=True,
                             text=True, check=True).stdout.strip()
    if "whopays" in sys.modules and os.environ.get("WHOPAYS_CODE_VERSION") != _commit:
        raise SystemExit("Restart the runtime before loading a different WhoPays build")
    # Pin our stack, but keep the platform's own numpy/pandas/pyarrow/scipy (like torch):
    # replacing them breaks the preinstalled Colab/Kaggle packages that depend on them.
    _keep = ("numpy==", "pandas==", "pyarrow==", "scipy==", "matplotlib==")
    with open(f"{_src}/requirements.lock") as f, open("/tmp/whopays-constraints.txt", "w") as g:
        g.writelines(line for line in f if not line.startswith(_keep))
    _sh(sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/whopays-constraints.txt",
        f"{_src}[gpu]")
    os.environ["WHOPAYS_CODE_VERSION"] = _commit

if MOUNT_DRIVE and PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["WHOPAYS_RESULTS"] = "/content/drive/MyDrive/whopays-results"

import whopays.env  # noqa: E402

whopays.env.print_env()

# 01 · FP16 baselines and the K1 gate

1. **Baselines:** FP16 Belebele, MGSM and FLORES NLL in 11 languages for each model in
   `MODELS`. Runs are resumable and synced to the HF results dataset, so a dead session
   loses at most one flush (50 items).
2. **K1 gate:** lm-evaluation-harness on the K1 model (1.5B), Belebele and MGSM, all 11
   languages, compared item by item with our runs. The gate passes when, for each task, the
   mean |ours − lm-eval| over languages is ≤ 1 point.
3. **Throughput:** measured GPU time, projected to the full benchmark and compared with the
   PRD's assumptions. More than 2× slower means re-planning the compute budget.

**Colab (about 2–2.5 h):** keep `MODELS` as the 1.5B only. **Kaggle:** add the 0.5B and 3B
(Save & Run All). Re-running skips finished work, including lm-eval's cached requests.

**Send back:** download the `.ipynb` after the run (File → Download → .ipynb).

In [ ]:
from whopays.grids import K1_MODEL, M0_MODELS

MODELS = [K1_MODEL]  # Kaggle: list(M0_MODELS)
RUN_LMEVAL = True
# Smoke tests only: lm-eval takes Belebele items in file order and we sort them by ID, so
# with a limit the two sides score different Belebele items and the comparison is empty.
LIMIT = None
if RUN_LMEVAL and K1_MODEL not in MODELS:
    raise ValueError("K1 requires its FP16 baselines: include K1_MODEL in MODELS")
print("models:", MODELS, "| all M0 models:", list(M0_MODELS))

## 1 · FP16 baselines

In [ ]:
import pandas as pd

import whopays
from whopays.config import load_jobs
from whopays.env import results_root
from whopays.grids import m0_fp16

JOBS = load_jobs(m0_fp16(MODELS, limit=LIMIT))
results = whopays.run(JOBS, stop_on_error=True)
status = pd.DataFrame([r.__dict__ for r in results])
print(status.groupby(["task", "status"]).size().unstack(fill_value=0))
print(status[status.status == "failed"][["run_id", "task", "lang", "error"]].to_string())

## 2 · K1 gate: lm-evaluation-harness on the same items

In [ ]:
from whopays import crosscheck
from whopays.io import hub_sync
from whopays.io.results import load_runs
from whopays.languages import CODES

LMEVAL_DIR = results_root() / f"lmeval-{K1_MODEL.split('/')[-1]}"
if RUN_LMEVAL:
    if not LMEVAL_DIR.exists():  # resume a previous session's request cache from the hub
        hub_sync.pull_runs(results_root(), [LMEVAL_DIR.name])
    crosscheck.run_lmeval(K1_MODEL, ["belebele", "mgsm"], CODES, LMEVAL_DIR, limit=LIMIT)

ours = load_runs(results_root(), [c for c in JOBS if c.model == K1_MODEL])
table = crosscheck.compare(ours, crosscheck.read_samples(LMEVAL_DIR))
pd.set_option("display.width", 160)
print(table.to_string(index=False))
print()
print(crosscheck.k1_verdict(table).to_string(index=False))

## 3 · Throughput vs the PRD's assumptions

In [ ]:
from whopays import throughput

rows = load_runs(results_root(), JOBS)
print(throughput.summary(rows).to_string(index=False))

# MGSM health per language: no-number outputs and outputs cut at the 256-token cap.
mg = rows[rows.task == "mgsm"]
if not mg.empty:
    print()
    print(mg.groupby(["model", "lang"])[["correct", "correct_ascii", "extraction_failed",
                                         "hit_max_tokens"]].mean().round(3).to_string())